# F Tuần 2 — Đánh giá pretrained vs fine-tuned trên bộ test công khai (tool-name, args, JSON validity, CI, McNemar)

**Mục tiêu:** đo **cùng một cách** hai mô hình trên `data/public/xlam_2k.eval.json` (200 bản ghi, mỗi bản ghi có
danh sách `tools` riêng):
1. `base` — `Qwen/Qwen2.5-0.5B-Instruct` zero-shot (không adapter);
2. `ft` — base + adapter `$OUT/full` bạn đã huấn luyện ở notebook 01.

Chuỗi công cụ (tất cả trong `docs/contracts/cli.md`):
`predict_toolcall.py` (sinh dự đoán: nạp fp16 theo compute capability, tools render qua `apply_chat_template`,
batch generate greedy) → `eval_toolcall.py` (chấm điểm theo `docs/contracts/eval_metric.md`) →
`bootstrap_ci.py` (CI 95% + McNemar theo cặp). Notebook **không** tự chấm điểm trong cell.

**Cần đọc trước:** `docs/contracts/eval_metric.md`, `docs/contracts/cli.md`, kết quả notebook 01 của chính bạn.

### Cell 1 — Thiết lập (ô chung của mọi notebook, copy nguyên văn từ `notebooks/_setup_snippet.md`)
Trên Colab: đọc `GITHUB_TOKEN` từ **Secrets** (biểu tượng chìa khóa ở thanh bên trái, bật *Notebook access*),
`git clone` repo private `thanhhao98/ChatSystem` (bỏ qua nếu đã có) rồi `chdir` vào đó. Trên máy cá nhân: đi lên từ
thư mục hiện tại đến khi gặp `docs/contracts/cli.md`. Ô đặt các biến `REPO`, `GIT_SHA`, `IN_COLAB`, `AUTHOR` và hàm
`run(cmd)`. Kaggle: token đọc từ *Add-ons → Secrets*. **Không bao giờ** in token hay `!cat .git/config` vào output.

In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if IN_COLAB or IN_KAGGLE:
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

### Cell 2 — Cài thư viện theo pins và kiểm tra GPU
Dòng đầu tiên in ra trên Colab miễn phí phải là:

```
Tesla T4 · 15 GB · (7, 5) · USE_BF16=False → fp16
```

- `(7, 5)` là *compute capability*; T4 < 8 nên **fp16**. Script huấn luyện tự chọn dtype theo đúng quy tắc này
  (biến môi trường `FORCE_FP16=1` ép fp16 trên GPU mới hơn để tái lập điều kiện T4).
- Nếu thấy `NO CUDA GPU`: *Runtime → Change runtime type → T4 GPU* rồi chạy lại từ Cell 1.
- Các phiên bản in ra phải trùng `requirements-train.txt`; nếu Colab đổi phiên bản `torch` thì ghi vào comment
  của task và mở issue kèm dòng phiên bản (pins được kiểm tra lại qua PR), **không** tự sửa pins.

In [ ]:
# Cell 2 — install the pinned training stack (Colab/Kaggle only) and probe the GPU.
import os
IN_KAGGLE = bool(globals().get("IN_KAGGLE")) or ((not IN_COLAB) and os.path.isdir("/kaggle/working"))
if IN_COLAB or IN_KAGGLE:
    !pip uninstall -y -q torchao
    !pip install -q -r requirements-train.txt
else:
    print("Local run: skipping pip install (use your own virtualenv built from requirements-train.txt).")

import importlib

import torch

FORCE_FP16 = os.environ.get("FORCE_FP16") == "1"
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
    GPU_CAP = torch.cuda.get_device_capability(0)
    # dtype is chosen by compute capability, NOT by torch.cuda.is_bf16_supported():
    # the T4 (7,5) only emulates bf16 and the training script would be slow / unstable.
    USE_BF16 = GPU_CAP[0] >= 8 and not FORCE_FP16
    GPU_LINE = (f"{GPU_NAME} · {GPU_GIB:.0f} GB · {GPU_CAP} · USE_BF16={USE_BF16} "
                f"→ {'bf16' if USE_BF16 else 'fp16'}")
else:
    GPU_NAME, GPU_GIB, GPU_CAP, USE_BF16 = None, 0.0, None, False
    GPU_LINE = "NO CUDA GPU (Colab: Runtime → Change runtime type → T4 GPU)"
print(GPU_LINE)

VERSIONS = {}
for _mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate"):
    try:
        VERSIONS[_mod] = importlib.import_module(_mod).__version__
    except Exception as exc:  # ImportError, or a broken CUDA extension on CPU-only hosts
        VERSIONS[_mod] = f"not installed ({type(exc).__name__})"
    print(f"{_mod:<13} {VERSIONS[_mod]}")
VERSIONS_LINE = " · ".join(f"{k} {v}" for k, v in VERSIONS.items())

### Cell 3 — Nơi lưu kết quả
Colab tắt phiên bất kỳ lúc nào (idle ~90 phút, hết quota ngày) và **xóa toàn bộ đĩa VM**. Vì vậy mọi kết quả
(checkpoint, adapter, log) ghi thẳng lên Google Drive: `OUT = /content/drive/MyDrive/ChatSystem/runs/<RUN_NAME>`.
Kaggle: `/kaggle/working/runs/<RUN_NAME>`; máy cá nhân: `runs/<RUN_NAME>` trong repo (đã gitignore).
Cache model Hugging Face để trên đĩa VM (mặc định) — không cần đưa lên Drive.

`RUN_NAME` phải **trùng** với notebook 01 để tìm thấy adapter tại `$OUT/full`. Kết quả tuần này ghi vào
`$OUT/results/public/`; khi nộp lên repo thì copy vào `results/public/` trong PR kèm dòng `results/RUNLOG.md`
và hàng `results/INDEX.md` (quy tắc "mọi con số phải truy vết được").

In [ ]:
# Cell 3 — where run outputs go. Colab: Google Drive (survives a disconnect); Kaggle: /kaggle/working; local: runs/.
RUN_NAME = "xlam2k_qwen05b"   # one folder per experiment; keep the SAME name across notebooks 01/02/03

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ChatSystem/runs/{RUN_NAME}"
elif IN_KAGGLE:
    OUT = f"/kaggle/working/runs/{RUN_NAME}"
else:
    OUT = str(REPO / "runs" / RUN_NAME)   # runs/ is gitignored
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT   # `!python … $OUT/…` lines and subprocesses see the same path

ADAPTER = f"{OUT}/full"
RES = f"{OUT}/results/public"
os.makedirs(RES, exist_ok=True)
os.environ["ADAPTER"], os.environ["RES"] = ADAPTER, RES
EVAL = "data/public/xlam_2k.eval.json"
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
_has_adapter = os.path.exists(f"{ADAPTER}/adapter_config.json")
print("ADAPTER =", ADAPTER, "(found)" if _has_adapter else "(KHÔNG thấy adapter_config.json — kiểm tra RUN_NAME / Drive)")
print("RES =", RES)
print("OUT =", OUT)

## Xem cờ của ba script

In [ ]:
!python training/predict_toolcall.py --help
!echo "----------------------------------------------------------------"
!python training/eval_toolcall.py --help
!echo "----------------------------------------------------------------"
!python training/bootstrap_ci.py --help

## Bước 1 — Dự đoán với mô hình base (zero-shot, không adapter)
`--backend transformers` nạp base ở fp16 (T4), render `tools` của **từng bản ghi** bằng `apply_chat_template`,
sinh greedy (`--temperature 0`) theo batch 8, tối đa 256 token mới. Dòng đầu file ra là header (`model`, `git_sha`,
`sha256_eval`, …) để scorer từ chối chấm nếu bộ eval đã đổi. Bản ghi được ghi dần vào `<out>.part` nên lượt chạy
bị ngắt vẫn giữ được phần đã xong.
Bỏ `--limit` để chạy đủ 200 bản ghi (vài phút trên T4). Chỉ dùng `--limit 20` khi muốn thử nhanh — và nhớ rằng
scorer chấm **toàn bộ** 200 bản ghi gold, bản ghi không có dự đoán tính là fail (`missing_prediction`).

In [ ]:
!python training/predict_toolcall.py --backend transformers --model-path Qwen/Qwen2.5-0.5B-Instruct --eval data/public/xlam_2k.eval.json --batch-size 8 --max-new-tokens 256 --out $RES/base_pred.jsonl

## Bước 2 — Dự đoán với adapter đã fine-tune
Cùng lệnh, thêm `--adapter $ADAPTER`. Mọi cờ khác **giữ nguyên** để hai lượt so sánh được.

In [ ]:
!python training/predict_toolcall.py --backend transformers --model-path Qwen/Qwen2.5-0.5B-Instruct --adapter $ADAPTER --eval data/public/xlam_2k.eval.json --batch-size 8 --max-new-tokens 256 --out $RES/ft_pred.jsonl

## Bước 3 — Chấm điểm hai file dự đoán (CPU)
`eval_toolcall.py` đối chiếu từng bản ghi với `expected_tool` / `expected_tool_alternates` / `expected_params`,
ghi `results.json` (schema trong `cli.md`: `accuracy`, `tool_name_acc`, `param_acc`, `params_exact`, `json_valid_rate`, `by_reason`, …)
và bản Markdown `--md` để dán báo cáo. `--group-by source` tách theo nguồn (bộ công khai chỉ có `public`; bộ SGOD
sau này có `human` / `gpt_aug`). Mã thoát 2 = header sha256 không khớp file gold cục bộ → **không** dùng
`--no-header-check` để "cho qua"; tìm hiểu vì sao bộ eval đổi.

In [ ]:
!python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json --pred $RES/base_pred.jsonl --out $RES/base.json --md $RES/base.md --group-by source
!echo "----------------------------------------------------------------"
!python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json --pred $RES/ft_pred.jsonl --out $RES/ft.json --md $RES/ft.md --group-by source

## Bước 4 — Bootstrap CI 95% và McNemar theo cặp (CPU)
`--run base=… --run ft=…` khai báo hai lượt; `--pair base ft` yêu cầu: CI của **hiệu** `acc_base − acc_ft` (paired
bootstrap, 2000 lần, seed 20260625), kết luận theo `--paired-diff-margin` (mặc định 5 pp — đúng ngưỡng H1 đã đăng ký
cho bộ SGOD), và McNemar exact hai phía (b, c, p). Lưu ý dấu: A = `base`, B = `ft`, nên **hiệu âm nghĩa là ft tốt hơn**;
dòng "H1" ở đây chỉ minh họa cách đọc, H1 thật là SLM-đã-fine-tune vs GPT trên `eval_v1`.

In [ ]:
!python training/bootstrap_ci.py --run base=$RES/base.json --run ft=$RES/ft.json --pair base ft --md $RES/bootstrap.md 2>&1 | tee $RES/bootstrap.log

## Bước 5 — Xem ba bản Markdown do script tạo

In [ ]:
# Render the Markdown files written by eval_toolcall.py / bootstrap_ci.py (nothing is recomputed here).
from IPython.display import Markdown, display

for name in ("base.md", "ft.md", "bootstrap.md"):
    path = f"{RES}/{name}"
    try:
        with open(path, encoding="utf-8") as f:
            display(Markdown(f"#### `{name}`\n\n" + f.read()))
    except FileNotFoundError:
        print(f"{path} chưa có — chạy Bước 3/4 trước.")

## Đọc số liệu như thế nào
- **Accuracy (strict pass) / tool acc:** một bản ghi *pass* khi **lời gọi đầu tiên** có tên tool nằm trong
  `{expected_tool} ∪ expected_tool_alternates` **và** tham số đạt ngưỡng; với `expected_tool = null` thì pass = không
  gọi tool; với `expected_permission = denied` thì pass = không gọi tool mà `tool_policy.json` cấm vai trò đó.
  `tool_name_acc` chỉ nhìn tên tool (bước 3 của thang điểm), `accuracy` đòi cả tham số.
- **Param acc:** trong các bản ghi đã đúng tool và có `expected_params`, tỷ lệ đạt ngưỡng ≥ ½ số khóa kỳ vọng có mặt
  trong `arguments`; dưới ngưỡng → `reason = missing_params`. `params_exact` khắt khe hơn: mọi khóa kỳ vọng có mặt và **bằng**
  giá trị gold (so sánh sau `str()`). Đây là nơi mô hình base thường mất điểm dù chọn đúng tool.
- **JSON valid rate:** `raw_text` chứa `<tool_call>{…}</tool_call>` parse được (hoặc cả chuỗi là JSON); sai định dạng →
  không có `predicted_tool` → `wrong_tool`. Fine-tune trên 1.6k hàng thường sửa lỗi này gần triệt để — đó là mức tăng
  dễ nhất, chưa phải "hiểu tool tốt hơn".
- **`by_reason`:** bảng phân rã lý do fail (`wrong_tool`, `missing_params`, `missing_prediction`, `api_error`, …; định
  nghĩa chính xác trong `docs/contracts/eval_metric.md`). Nhìn vào đây trước khi kết luận gì về mô hình.
- **CI 95% (bootstrap percentile, 2000 lần, seed 20260625):** với n = 200, nửa rộng CI ≈ ±6–7 pp — hai CI chồng nhau
  chưa nói được gì. Cái để kết luận là **CI của hiệu** (paired: resample cùng chỉ số cho cả hai mô hình); CI của hiệu
  không chứa 0 → khác biệt thật.
- **McNemar (b, c, p):** chỉ đếm bản ghi hai mô hình **bất đồng**: b = base pass & ft fail, c = base fail & ft pass;
  p là exact two-sided binomial trên (b, c). p < 0.05 → khác biệt có ý nghĩa; luôn báo cả ba số, không chỉ p.

## Tạo báo cáo
Dán khối in ra vào bình luận task ClickUp *F Tuần 2*. Bảng đọc từ `base.json` / `ft.json`; dòng CI/McNemar đọc từ
`bootstrap.log` (stdout của `bootstrap_ci.py`); không gõ tay.

In [ ]:
# "## Tạo báo cáo" — prints ONE Markdown block to paste as the ClickUp task comment.
# Every number is read from files the scripts wrote; anything missing prints as n/a (never typed by hand).
import datetime
import json
import re
import subprocess


def _read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def _grep(path, pattern, max_lines=3):
    """Last `max_lines` lines of `path` matching `pattern` (case-insensitive); [] when the file is missing."""
    try:
        with open(path, errors="replace") as f:
            hits = [ln.rstrip() for ln in f if re.search(pattern, ln, re.I)]
        return hits[-max_lines:]
    except FileNotFoundError:
        return []


def _ls(path):
    r = subprocess.run(["ls", "-la", path], capture_output=True, text=True)
    return r.stdout.strip() or f"(missing: {path})"


def _log_history(path):
    """log_history.json is trainer.state.log_history: a list of dicts (accept {"log_history": [...]} too)."""
    hist = _read_json(path)
    if isinstance(hist, dict):
        hist = hist.get("log_history", [])
    return hist or []


ENV_LINE = "Colab" if IN_COLAB else ("Kaggle" if globals().get("IN_KAGGLE") else "local")
TODAY = datetime.date.today().isoformat()

import hashlib

TASK = "Tuần 2 — Đánh giá pretrained vs fine-tuned trên test công khai (tool-name, args, JSON validity, bootstrap CI, McNemar)"


def _acc_row(label, res):
    if not res:
        return f"| {label} | n/a | n/a | n/a | n/a | n/a | n/a | n/a | n/a |"
    n, passed = res.get("n"), res.get("passed")
    if n is None or passed is None:
        rows = res.get("results", [])
        n, passed = len(rows), sum(1 for r in rows if r.get("pass"))
    acc = res.get("accuracy", 100.0 * passed / n if n else float("nan"))
    by_reason = res.get("by_reason") or {}
    top = ", ".join(f"{k}={v}" for k, v in sorted(by_reason.items(), key=lambda kv: -kv[1]) if k != "ok")
    tool_acc = res.get("tool_name_acc", res.get("tool_acc", "n/a"))
    return (f"| {label} | {n} | {passed} | {acc:.1f} | {tool_acc} | {res.get('param_acc', 'n/a')} | "
            f"{res.get('params_exact', 'n/a')} | {res.get('json_valid_rate', 'n/a')} | {top or '—'} |")


base_res = _read_json(f"{RES}/base.json")
ft_res = _read_json(f"{RES}/ft.json")
ci_lines = _grep(f"{RES}/bootstrap.log", r"^\s*(base|ft)\s+\d|diff = ", 4)
mcnemar_lines = _grep(f"{RES}/bootstrap.log", r"McNemar", 2) or ["n/a (chưa chạy Bước 4)"]
try:
    with open(EVAL, "rb") as f:
        eval_sha = hashlib.sha256(f.read()).hexdigest()[:8]
except FileNotFoundError:
    eval_sha = "n/a"

CMDS = [
    f"python training/predict_toolcall.py --backend transformers --model-path {BASE_MODEL} --eval {EVAL} --batch-size 8 --max-new-tokens 256 --out {RES}/base_pred.jsonl",
    f"python training/predict_toolcall.py --backend transformers --model-path {BASE_MODEL} --adapter {ADAPTER} --eval {EVAL} --batch-size 8 --max-new-tokens 256 --out {RES}/ft_pred.jsonl",
    f"python training/eval_toolcall.py --gold {EVAL} --pred {RES}/base_pred.jsonl --out {RES}/base.json --md {RES}/base.md --group-by source",
    f"python training/eval_toolcall.py --gold {EVAL} --pred {RES}/ft_pred.jsonl --out {RES}/ft.json --md {RES}/ft.md --group-by source",
    f"python training/bootstrap_ci.py --run base={RES}/base.json --run ft={RES}/ft.json --pair base ft --md {RES}/bootstrap.md",
]

lines = [
    f"## Báo cáo {TASK} — {TODAY} — {AUTHOR}",
    f"- Notebook: `notebooks/finetune/02_eval_toolcalling.ipynb` @ `{GIT_SHA}` · môi trường: {ENV_LINE} · RUN_NAME `{RUN_NAME}`",
    f"- Trục đo: eval `{EVAL}` @ sha256 `{eval_sha}` · tools theo từng bản ghi · preamble v0 · scorer strict (first call) · decoding greedy",
    f"- GPU: {GPU_LINE}",
    f"- Phiên bản: {VERSIONS_LINE}",
    f"- Mô hình: base = `{BASE_MODEL}` (zero-shot) · ft = base + adapter `{ADAPTER}`",
    "",
    "| run | n | passed | accuracy % | tool_name_acc % | param_acc % | params_exact % | json_valid % | fail reasons |",
    "|---|---|---|---|---|---|---|---|---|",
    _acc_row("base", base_res),
    _acc_row("ft", ft_res),
    "",
    "- Bootstrap CI 95% (từng run) và CI của hiệu base − ft (paired):",
    "```", *(ci_lines or ["n/a (chưa chạy Bước 4)"]), "```",
    "- McNemar (base vs ft): " + " | ".join(f"`{h.strip()}`" for h in mcnemar_lines),
    "- Lệnh đã chạy:",
    "```", *CMDS, "```",
    f"- Tệp kết quả: `{RES}/` (base_pred.jsonl, ft_pred.jsonl, base.json, ft.json, base.md, ft.md, bootstrap.md) → copy vào `results/public/` của repo khi nộp PR, thêm dòng RUNLOG + hàng INDEX.",
]
print("\n".join(lines))

## Bước nâng cao (không bắt buộc)
Muốn thấy adapter chạy như một **dịch vụ** (giống cách backend gọi SLM) thay vì nạp trực tiếp bằng transformers:
mở `notebooks/finetune/03_serve_vllm_colab.ipynb` — phục vụ base + LoRA bằng vLLM (`--dtype half`, hermes tool
parser) ngay trên T4, gọi lại `predict_toolcall.py --backend openai` qua OpenAI-compatible API, đo p50 latency.
Nếu vLLM không cài được trên Colab thì **kết quả của notebook này đã đủ** làm bằng chứng độ chính xác; việc phục vụ
adapter chạy trên máy GPU của hạ tầng tham chiếu (`docs/serving.md`).